# Module 10: Bias, Fairness & Sustainability Audit

**Unit E · Week 8** · Track 2 — Visualization, Analytic Hubs & Responsible AI

Auditing the Module 8 model for group-wise fairness by province, then drafting a mitigation rather than stopping at detection.

## Learning objectives

- **Advanced** _Analyze/Evaluate_ — Run a group-wise fairness/bias audit on a model and draft mitigations.
- **Advanced** _Analyze/Evaluate_ — Evaluate a pipeline or hub end-to-end for sustainability and accountability risk.

## Setup

This notebook reads `../../data/processed/track2_dataset.csv`, built by
`data/make_sample_data.py`. It is **synthetic** data shaped like the real
NISR/HDX handoff described in the course site's Chapter 3 — swap in a real
extract by re-pointing the path below once you have one. See
`data/README.md` for where to get real data and exactly what to rename.


In [1]:
import pandas as pd

DATA_PATH = "../../data/processed/track2_dataset.csv"
df = pd.read_csv(DATA_PATH, parse_dates=["date"])
print(f"{len(df):,} rows · {df['district'].nunique()} districts · "
      f"{df['date'].min().date()} to {df['date'].max().date()}")
df.head()

240 rows · 10 districts · 2023-01-01 to 2024-12-01


,date,province,district,district_pcode,indicator,value,feature_1,feature_2,outcome
0,2023-01-01,Kigali City,Gasabo,SIM-GAS,sample_wellbeing_index,39.37,82.01,19.22,1
1,2023-02-01,Kigali City,Gasabo,SIM-GAS,sample_wellbeing_index,45.24,49.99,17.62,1
2,2023-03-01,Kigali City,Gasabo,SIM-GAS,sample_wellbeing_index,41.16,46.66,19.93,0
3,2023-04-01,Kigali City,Gasabo,SIM-GAS,sample_wellbeing_index,42.91,49.17,20.35,0
4,2023-05-01,Kigali City,Gasabo,SIM-GAS,sample_wellbeing_index,42.05,51.04,20.35,0


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from fairlearn.metrics import MetricFrame, demographic_parity_difference
from sklearn.metrics import accuracy_score

X = df[["feature_1", "feature_2"]]
y = df["outcome"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = LogisticRegression().fit(X_train, y_train)
preds = model.predict(X_test)

sub = df.loc[X_test.index, "province"]
mf = MetricFrame(metrics=accuracy_score, y_true=y_test, y_pred=preds, sensitive_features=sub)
print("Accuracy by province:\n", mf.by_group)

dpd = demographic_parity_difference(y_test, preds, sensitive_features=sub)
print(f"\nDemographic parity difference: {dpd:.3f}")

Accuracy by province:
 province
Eastern        0.666667
Kigali City    0.823529
Northern       0.846154
Southern       0.818182
Western        0.500000
Name: accuracy_score, dtype: float64

Demographic parity difference: 0.100


**Mitigation note (fill in after running):** if accuracy or the parity gap above is wide across provinces, name at least one mitigation, e.g. a province-aware decision threshold, collecting more training data from the underperforming province, or dropping the model in favor of the simpler Module 7 forecast for that province specifically.

## Your turn

Fairness audit report on the Module 8 model, disaggregated by province, with at least one drafted mitigation.

**Formative assessment.** Graded audit report. The second, advanced-level formative touchpoint for the Responsible AI domain, immediately ahead of the capstone.